# Part 1 – Foundry Lab (Workflow‑First Multi‑Agent Intelligence)

This notebook uses **Microsoft Agent Framework (Python)** to orchestrate multiple **Azure AI Foundry Agent Service** agents in a **sequential workflow** using `WorkflowBuilder`.

✅ Assumption: **`ProductInventoryManagerAgent` already exists in Azure AI Foundry** (created in the portal UI).

We will:
1. Authenticate with Azure (`az login` + `AzureCliCredential`)
2. Reuse the existing Foundry agent by **agent ID**
3. Create additional *ephemeral* agents for this lab (CustomerInsight, SalesInsight, MarketingAction, Coordinator)
4. Build a **sequential pipeline** workflow with `WorkflowBuilder(start_executor=...)` and `add_edge(...)`
5. Run the workflow and print final outputs

References (docs):
- “Agents in Workflows” shows using `WorkflowBuilder` + `AzureAIAgentClient` + `AzureCliCredential` and emphasizes async context management.
- “Azure AI Foundry Agents” shows how to use an **existing agent by ID** with `AzureAIAgentClient`.
- “Workflow Builder & Execution” shows `WorkflowBuilder(start_executor=...)` + `add_edge(...)` + `build()` in Python.


## 0) Install dependencies

Uncomment and run if needed:


In [ ]:
# %pip install -U agent-framework --pre
# %pip install -U agent-framework-azure-ai --pre
# %pip install -U azure-identity


## 1) Set environment variables

Set these in your shell or a `.env` (do not commit).

Required:
- `AZURE_AI_PROJECT_ENDPOINT`
- `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- `PRODUCT_INVENTORY_MANAGER_AGENT_ID` (existing Foundry agent ID from portal)

Example:
```bash
export AZURE_AI_PROJECT_ENDPOINT="https://<aiservices>.services.ai.azure.com/api/projects/<project>"
export AZURE_AI_MODEL_DEPLOYMENT_NAME="gpt-4o-mini"
export PRODUCT_INVENTORY_MANAGER_AGENT_ID="<existing-agent-id>"
```


In [ ]:
import os

required = [
    'AZURE_AI_PROJECT_ENDPOINT',
    'AZURE_AI_MODEL_DEPLOYMENT_NAME',
    'PRODUCT_INVENTORY_MANAGER_AGENT_ID'
]
missing = [k for k in required if not os.getenv(k)]
print('Missing env vars:', missing)


## 2) Build the agents (1 existing + 4 ephemeral)

We reuse the existing `ProductInventoryManagerAgent` by supplying its **agent ID**.
For other roles, we create ephemeral agents for the lab run.

We use `AsyncExitStack` to correctly manage async context (credential + agents).


In [ ]:
import asyncio
from contextlib import AsyncExitStack

from azure.identity.aio import AzureCliCredential

from agent_framework import Agent
from agent_framework.azure import AzureAIAgentClient


async def build_agents(stack: AsyncExitStack):
    # Auth: uses az login
    credential = await stack.enter_async_context(AzureCliCredential())

    project_endpoint = os.environ['AZURE_AI_PROJECT_ENDPOINT']
    model_deployment = os.environ['AZURE_AI_MODEL_DEPLOYMENT_NAME']

    # Existing Foundry agent by ID (created in portal UI)
    inventory_agent_id = os.environ['PRODUCT_INVENTORY_MANAGER_AGENT_ID']

    inventory_agent = await stack.enter_async_context(
        Agent(
            name='ProductInventoryManagerAgent',
            instructions=(
                'You are ProductInventoryManagerAgent. ' 
                'Analyze inventory risk (stockout/excess), flag fast/slow movers, ' 
                'and return a concise JSON summary with key SKUs, rationale, and suggested actions.'
            ),
            chat_client=AzureAIAgentClient(
                async_credential=credential,
                project_endpoint=project_endpoint,
                model_deployment_name=model_deployment,
                agent_id=inventory_agent_id,
            ),
        )
    )

    # Ephemeral agents for the lab (created for this run)
    customer_insight = await stack.enter_async_context(
        Agent(
            name='CustomerInsightAgent',
            instructions=(
                'You are CustomerInsightAgent. Given sales + inventory findings, '
                'infer customer segment impacts (e.g., high-value customers affected by stockouts), '
                'and propose 2-3 customer-facing insights in JSON.'
            ),
            chat_client=AzureAIAgentClient(
                async_credential=credential,
                project_endpoint=project_endpoint,
                model_deployment_name=model_deployment,
            ),
        )
    )

    sales_insight = await stack.enter_async_context(
        Agent(
            name='SalesInsightAgent',
            instructions=(
                'You are SalesInsightAgent. Translate inventory + customer insights into '
                'revenue risk/opportunity (upsell/cross-sell, replenishment urgency), '
                'and output a ranked list of top opportunities in JSON.'
            ),
            chat_client=AzureAIAgentClient(
                async_credential=credential,
                project_endpoint=project_endpoint,
                model_deployment_name=model_deployment,
            ),
        )
    )

    marketing_action = await stack.enter_async_context(
        Agent(
            name='MarketingActionAgent',
            instructions=(
                'You are MarketingActionAgent. Propose marketing actions aligned to '
                'inventory reality (avoid promoting out-of-stock, push excess stock), '
                'and produce an action plan with channels, timing, and guardrails in JSON.'
            ),
            chat_client=AzureAIAgentClient(
                async_credential=credential,
                project_endpoint=project_endpoint,
                model_deployment_name=model_deployment,
            ),
        )
    )

    coordinator = await stack.enter_async_context(
        Agent(
            name='ZavaCoordinatorAgent',
            instructions=(
                'You are ZavaCoordinatorAgent. Convert the user request into a structured brief '
                'for downstream agents. Maintain a running JSON object called brief with fields: '
                'goal, constraints, data_notes, required_outputs.'
            ),
            chat_client=AzureAIAgentClient(
                async_credential=credential,
                project_endpoint=project_endpoint,
                model_deployment_name=model_deployment,
            ),
        )
    )

    return coordinator, inventory_agent, customer_insight, sales_insight, marketing_action


async def quick_smoke_test(agent: Agent):
    resp = await agent.run('Say hello in one sentence.')
    print(resp.text)


async def main_build_only():
    async with AsyncExitStack() as stack:
        agents = await build_agents(stack)
        print('Agents ready:', [a.name for a in agents])
        await quick_smoke_test(agents[0])


# Uncomment to test agent connectivity
# asyncio.run(main_build_only())


## 3) Build the sequential workflow with WorkflowBuilder

We create a directed pipeline:

`ZavaCoordinatorAgent` → `ProductInventoryManagerAgent` → `CustomerInsightAgent` → `SalesInsightAgent` → `MarketingActionAgent`

The docs show the Python pattern `WorkflowBuilder(start_executor=...)` + `add_edge(...)` + `build()`.

⚠️ Note: Some previews have small signature differences. This notebook includes a safe fallback: if the constructor form fails, it will create an empty builder and call `set_start_executor`.


In [ ]:
import inspect
from agent_framework import WorkflowBuilder


def build_workflow(builder_cls, start_executor, edges):
    """Create a WorkflowBuilder in a version-tolerant way and return workflow.

    edges: list of (source, target)
    """
    try:
        # Preferred modern pattern
        builder = builder_cls(start_executor=start_executor)
    except TypeError:
        # Fallback for variants that require setting start executor separately
        builder = builder_cls()
        if hasattr(builder, 'set_start_executor'):
            builder.set_start_executor(start_executor)
        else:
            raise

    for src, dst in edges:
        builder.add_edge(src, dst)

    return builder.build()


async def main_run_workflow(user_prompt: str):
    async with AsyncExitStack() as stack:
        coordinator, inventory, customer, sales, marketing = await build_agents(stack)

        edges = [
            (coordinator, inventory),
            (inventory, customer),
            (customer, sales),
            (sales, marketing),
        ]

        workflow = build_workflow(WorkflowBuilder, coordinator, edges)
        result = await workflow.run(user_prompt)

        # Most workflow examples use get_outputs() to retrieve final outputs
        try:
            outputs = result.get_outputs()
        except Exception:
            outputs = None

        print('=== WORKFLOW OUTPUTS ===')
        print(outputs if outputs is not None else result)


prompt = (
    'Zava: Using our sales + product + customer dataset, produce a weekly brief: '
    '1) inventory risks (stockout/excess), 2) customer impact insights, '
    '3) sales opportunities, 4) marketing actions. Return JSON.'
)

# Run the workflow
# asyncio.run(main_run_workflow(prompt))


## 4) Next steps (extend this lab)

- Add **structured output enforcement** using JSON schema / Pydantic in the agent instructions (common pattern in Agent Framework docs).
- Add **conditional routing** (for example, route to a different marketing agent if inventory is constrained).
- Add **MCP tools** for CRM or SQL/Cosmos lookups once you move to Part 2 (code-first).
